# 🧠 Multimodal Deepfake Detection — Web UI
**Presented at ICRETEM 2026** | Avg Accuracy: **86.6%** | Macro-AUC: **0.980** (FakeAVCeleb)

This notebook launches a **Flask** web server tunnelled through **ngrok**, providing a production-grade UI for deepfake inference with Grad-CAM XAI and Gemini forensic reports.

### Steps
1. Run **Cell 1** — Install dependencies  
2. Run **Cell 2** — Mount Google Drive  
3. Run **Cell 3** — Set your API keys  
4. Run **Cell 4** — Model definition  
5. Run **Cell 5** — Inference + XAI functions  
6. Run **Cell 6** — Flask app + ngrok launch  

In [1]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies
# ─────────────────────────────────────────────────────────────
!pip install -q retina-face librosa opencv-python ffmpeg-python \
    torch torchvision google-generativeai \
    flask flask-cors pyngrok Pillow matplotlib scikit-learn

In [2]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — Mount Google Drive
# ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


In [3]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Configuration & API keys
# ─────────────────────────────────────────────────────────────
import os

# ── Paths ────────────────────────────────────────────────────
CHECKPOINT_PATH = "/content/drive/MyDrive/Deepfake/joint_model_checkpoint.pth"

# ── API Keys ─────────────────────────────────────────────────
GEMINI_API_KEY = ""          # ← Leave blank; key is entered by the user in the web UI
NGROK_AUTH_TOKEN = ""        # ← https://dashboard.ngrok.com/get-started/your-authtoken

# ── Model constants ───────────────────────────────────────────
FRAME_SIZE    = 224
NUM_FRAMES    = 16
NUM_AUDIO_SEG = 16
SR            = 16000
N_MELS        = 128
TARGET_MEL_T  = 32
UPLOAD_TIMEOUT_S = 60   # 1 min hard timeout per inference
MAX_VIDEO_MB     = 200   # reject files larger than this

CLASS_NAMES = {
    0: "Real Video + Real Audio",
    1: "Fake Video + Real Audio",
    2: "Real Video + Fake Audio",
    3: "Fake Video + Fake Audio"
}

# Project metrics (hardcoded from paper)
PAPER_METRICS = {
    "datasets": [
        {"name": "FakeAVCeleb", "accuracy": 90.4, "macro_auc": 0.980, "eer": 0.9},
        {"name": "LAV-DF",      "accuracy": 86.3, "macro_auc": 0.949, "eer": 0.0},
        {"name": "FaceForensics++ C23", "accuracy": 83.1, "macro_auc": 0.852, "eer": 17.9},
        {"name": "Average",     "accuracy": 86.6, "macro_auc": None,  "eer": None}
    ],
    "f1_scores": [
        {"class": "Real / Real",             "f1": 92.9},
        {"class": "Fake Video / Real Audio",  "f1": 89.3},
        {"class": "Real Video / Fake Audio",  "f1": 89.6},
        {"class": "Fake / Fake",              "f1": 91.2}
    ],
    "training": {
        "optimizer": "Adam (lr=1e-4)",
        "loss": "Focal Loss (γ=2.0) + Label Smoothing (0.1)",
        "epochs": "10 (early stopping)",
        "mixed_precision": "AMP fp16",
        "grad_clipping": "Max norm 1.0"
    }
}

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Config loaded | Device: {device}')
print(f'   Checkpoint: {CHECKPOINT_PATH}')

✅ Config loaded | Device: cuda
   Checkpoint: /content/drive/MyDrive/Deepfake/joint_model_checkpoint.pth


In [10]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — Model architecture (identical to training)
# ─────────────────────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights

class TemporalFrameDropout(nn.Module):
    def __init__(self, p=0.2):
        super().__init__()
        self.p = p
    def forward(self, x):
        if not self.training:
            return x
        mask = (torch.rand(x.size(0), x.size(1), 1, device=x.device) > self.p).float()
        return x * mask

class VisualEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        base = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(*list(base.children())[:-1])
    def forward(self, x):
        B, T, C, H, W = x.shape
        f = self.features(x.view(B*T, C, H, W))
        return f.view(B, T, -1)

class TemporalTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=512, nhead=8, batch_first=True), 2)
    def forward(self, x):
        return self.transformer(x).mean(1)

class AudioEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1))
        )
        self.fc = nn.Linear(128, 256)
    def forward(self, x):
        B, T, C, H, W = x.shape
        return self.fc(self.conv(x.view(B*T,C,H,W)).view(B,T,-1).mean(1))

class CrossModalTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.v_proj = nn.Linear(512, 512)
        self.a_proj = nn.Linear(256, 512)
        self.cross  = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)
    def forward(self, v, a):
        v = self.v_proj(v).unsqueeze(1)
        a = self.a_proj(a).unsqueeze(1)
        fused, _ = self.cross(v, a, a)
        return fused.squeeze(1)

class AVSyncFeature(nn.Module):
    def __init__(self):
        super().__init__()
        self.audio_proj = nn.Linear(256, 512)
    def forward(self, v, a):
        a = self.audio_proj(a)
        v = F.normalize(v, dim=1)
        a = F.normalize(a, dim=1)
        return (v * a).sum(dim=1, keepdim=True)

class MultimodalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual   = VisualEncoder()
        self.drop     = TemporalFrameDropout()
        self.temporal = TemporalTransformer()
        self.audio    = AudioEncoder()
        self.cross    = CrossModalTransformer()
        self.sync     = AVSyncFeature()
        self.fc = nn.Sequential(
            nn.Linear(512 + 1, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 4)
        )
    def forward(self, frames, audio):
        v     = self.visual(frames)
        v     = self.drop(v)
        v     = self.temporal(v)
        a     = self.audio(audio)
        fused = self.cross(v, a)
        sync  = self.sync(v, a)
        return self.fc(torch.cat([fused, sync], dim=1))

# ── Load checkpoint ──────────────────────────────────────────
model = MultimodalModel().to(device)
ckpt  = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✅ Model loaded — avg accuracy: {ckpt["avg_acc"]*100:.2f}%')
print(f'   Best epoch: {ckpt["epoch"]}')
for name, acc in ckpt.get('accs', {}).items():
    print(f'   {name}: {acc*100:.1f}%')

✅ Model loaded — avg accuracy: 86.59%
   Best epoch: 8
   FakeAVCeleb: 90.4%
   LAV-DF: 86.2%
   FF++: 83.1%


In [13]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Preprocessing, XAI & Gemini inference functions
# ─────────────────────────────────────────────────────────────
import cv2, numpy as np, librosa, ffmpeg, time, base64, json, io
import matplotlib
matplotlib.use('Agg')        # headless — required for Flask
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import google.generativeai as genai
from retinaface import RetinaFace
from PIL import Image

# ── Configure Gemini ─────────────────────────────────────────
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.5-flash-lite')

# ── Video preprocessing ──────────────────────────────────────
def detect_face_box(frame):
    try:
        faces = RetinaFace.detect_faces(frame)
        if isinstance(faces, dict) and len(faces) > 0:
            largest = max(
                faces.values(),
                key=lambda x: (x['facial_area'][2]-x['facial_area'][0]) *
                              (x['facial_area'][3]-x['facial_area'][1])
            )
            return largest['facial_area']
    except Exception:
        pass
    return None

def extract_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {video_path}')
    frames, face_box, frame_count = [], None, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1
        if face_box is None and frame_count <= 30:
            face_box = detect_face_box(frame)
        if face_box is not None:
            x1, y1, x2, y2 = [int(v) for v in face_box]
            h, w = frame.shape[:2]
            crop = frame[max(0,y1):min(h,y2), max(0,x1):min(w,x2)]
        else:
            h, w = frame.shape[:2]
            s = min(h, w) // 2
            crop = frame[h//2-s:h//2+s, w//2-s:w//2+s]
        if crop.size == 0:
            crop = frame
        frames.append(cv2.resize(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB),
                                  (FRAME_SIZE, FRAME_SIZE)))
        if len(frames) >= NUM_FRAMES:
            break
    cap.release()
    if len(frames) == 0:
        raise ValueError('No frames could be extracted from the video.')
    while len(frames) < NUM_FRAMES:
        frames.append(frames[-1])
    return frames

def extract_audio(video_path, tmp_path='/tmp/tmp_audio.wav'):
    try:
        ffmpeg.input(video_path).output(
            tmp_path, ac=1, ar=SR
        ).overwrite_output().run(quiet=True)
        if os.path.exists(tmp_path) and os.path.getsize(tmp_path) > 1000:
            return tmp_path, True
    except Exception:
        pass
    return tmp_path, False

def process_audio(path, has_audio=True, video_path=None):
    if not has_audio:
        if video_path:
            cap   = cv2.VideoCapture(video_path)
            fps   = cap.get(cv2.CAP_PROP_FPS) or 25
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            audio = np.zeros(int(SR * total / fps), dtype=np.float32)
        else:
            audio = np.zeros(SR * 5, dtype=np.float32)
        sr = SR
    else:
        audio, sr = librosa.load(path, sr=SR)
    if len(audio) < NUM_AUDIO_SEG:
        audio = np.pad(audio, (0, NUM_AUDIO_SEG - len(audio)))
    seg_len = len(audio) // NUM_AUDIO_SEG
    feats   = []
    for i in range(NUM_AUDIO_SEG):
        seg = audio[i*seg_len:(i+1)*seg_len]
        mel = librosa.power_to_db(
            librosa.feature.melspectrogram(y=seg, sr=sr, n_mels=N_MELS))
        mel = np.pad(
            mel, ((0,0),(0,max(0, TARGET_MEL_T-mel.shape[1]))),
            mode='constant')[:, :TARGET_MEL_T]
        feats.append(mel)
    return np.array(feats)

# ── XAI ──────────────────────────────────────────────────────
def compute_gradcam(model, frames_t, audio_t, target_class):
    gradients, activations = [], []
    target_layer = model.visual.features[-2]
    def fwd_hook(m, i, o): activations.append(o)
    def bwd_hook(m, i, o): gradients.append(o[0])
    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)
    frames_input = frames_t.clone().requires_grad_(True)
    logits = model(frames_input, audio_t)
    model.zero_grad()
    logits[0, target_class].backward()
    h1.remove(); h2.remove()
    grads   = gradients[0]
    acts    = activations[0]
    weights = grads.mean(dim=(2,3), keepdim=True)
    cam_raw = F.relu((weights * acts).sum(dim=1))
    cams    = cam_raw.view(NUM_FRAMES, *cam_raw.shape[1:]).detach().cpu().numpy()
    result  = []
    for c in cams:
        c = c - c.min()
        c = c / (c.max() + 1e-8)
        c = cv2.resize(c, (FRAME_SIZE, FRAME_SIZE))
        result.append(c)
    return np.array(result)

def compute_temporal_attention(model, frames_t, audio_t):
    frame_features = []
    def hook_fn(m, i, o): frame_features.append(o.detach().cpu())
    h = model.visual.register_forward_hook(hook_fn)
    with torch.no_grad():
        model(frames_t, audio_t)
    h.remove()
    if not frame_features:
        return np.ones(NUM_FRAMES) / NUM_FRAMES
    feats  = frame_features[0].squeeze(0)
    d      = feats.shape[-1]
    scores = torch.matmul(feats, feats.T) / (d ** 0.5)
    scores = torch.softmax(scores, dim=-1)
    attn   = scores.mean(dim=0).numpy()
    attn   = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)
    return attn

def compute_audio_saliency(audio_np):
    energies = np.array([np.abs(seg).mean() for seg in audio_np])
    energies = (energies - energies.min()) / (energies.max() - energies.min() + 1e-8)
    return energies

# ── Visualize XAI → PNG bytes (base64) ───────────────────────
def generate_xai_image(frames, gradcams, temporal_attn, audio_sal, probs, pred, filename):
    fig = plt.figure(figsize=(24, 14))
    fig.patch.set_facecolor('#0f0f0f')
    gs  = gridspec.GridSpec(3, 8, figure=fig, hspace=0.4, wspace=0.3)
    cmap_hot = plt.cm.jet

    for i in range(8):
        ax = fig.add_subplot(gs[0, i])
        ax.imshow(frames[i])
        ax.set_title(f'F{i+1}', color='white', fontsize=7)
        ax.axis('off')

    for i in range(8):
        ax = fig.add_subplot(gs[1, i])
        cam_color = cmap_hot(gradcams[i])[:,:,:3]
        overlay   = np.clip(0.45 * frames[i]/255 + 0.55 * cam_color, 0, 1)
        ax.imshow(overlay)
        ax.set_title(f'GradCAM F{i+1}', color='#ff6666', fontsize=7)
        ax.axis('off')

    ax3 = fig.add_subplot(gs[2, :4])
    colors = ['#00ff99' if s > 0.5 else '#ff4444' for s in temporal_attn]
    ax3.bar(range(NUM_FRAMES), temporal_attn, color=colors, edgecolor='none')
    ax3.set_facecolor('#1a1a1a')
    ax3.set_title('Temporal Attention per Frame', color='white', fontsize=10)
    ax3.set_xlabel('Frame', color='#aaaaaa')
    ax3.set_ylabel('Attention Score', color='#aaaaaa')
    ax3.tick_params(colors='#aaaaaa')
    for spine in ax3.spines.values(): spine.set_edgecolor('#333333')

    ax4 = fig.add_subplot(gs[2, 4:])
    ax4.fill_between(range(NUM_AUDIO_SEG), audio_sal, alpha=0.7, color='#4488ff')
    ax4.plot(range(NUM_AUDIO_SEG), audio_sal, color='#88bbff', linewidth=1.5)
    ax4.set_facecolor('#1a1a1a')
    ax4.set_title('Audio Segment Saliency', color='white', fontsize=10)
    ax4.set_xlabel('Segment', color='#aaaaaa')
    ax4.set_ylabel('Energy', color='#aaaaaa')
    ax4.tick_params(colors='#aaaaaa')
    for spine in ax4.spines.values(): spine.set_edgecolor('#333333')

    emoji = '🚨' if pred in [1,2,3] else '✅'
    fig.suptitle(
        f'{emoji}  {CLASS_NAMES[pred]}  —  {probs[pred]*100:.1f}% confidence  |  {filename}',
        color='white', fontsize=13, fontweight='bold', y=0.98
    )

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=120, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')

# ── Gemini forensic report ───────────────────────────────────
def explain_with_gemini(probs, pred, temporal_attn, audio_sal, gradcams, filename, api_key=""):
    top_frames      = np.argsort(temporal_attn)[::-1][:4] + 1
    suspicious_segs = np.where(audio_sal > 0.6)[0] + 1
    gradcam_mean    = gradcams.mean(axis=(1,2))
    hot_frames      = np.argsort(gradcam_mean)[::-1][:4] + 1
    av_sync_score   = float(probs[0])
    video_fake_prob = float(probs[1] + probs[3])
    audio_fake_prob = float(probs[2] + probs[3])

    prompt = f"""You are a senior Digital Forensic Investigator. Translate the following deepfake detection analysis into a clear, jargon-free report suitable for a courtroom or news article.

VIDEO FILE: {filename}
OVERALL VERDICT: {CLASS_NAMES[pred]}
CONFIDENCE: {probs[pred]*100:.1f}%

TECHNICAL DATA:
1. Visual Evidence: Grad-CAM (visual focus map) peaked at frames {list(hot_frames)} with a video manipulation probability of {video_fake_prob*100:.1f}%.
2. Audio Evidence: Audio saliency detected anomalies in segments {list(suspicious_segs) if len(suspicious_segs) > 0 else '[none above threshold]'} with an audio fake probability of {audio_fake_prob*100:.1f}%.
3. Sync Evidence: There is a {(1-av_sync_score)*100:.1f}% mismatch between mouth movements and the voice track (AV lip-sync score).

INSTRUCTIONS:
- Call Grad-CAM 'visual focus analysis' or 'attention mapping'.
- Call AV Sync 'lip-sync coherence check'.
- Describe specific frames as if a forensic examiner is looking at them.
- Use a serious, professional but accessible tone. Avoid bullet points — write in flowing prose.
- Weave numbers into sentences naturally.

Write exactly 3 detailed paragraphs (Visual Analysis, Audio Analysis, Sync & Overall Assessment). End with a one-sentence **Final Verdict** in bold. Do not truncate — write the full report."""

    if not api_key:
        raise RuntimeError("Gemini API key not provided. Please enter your key in the web UI.")
    genai.configure(api_key=api_key)
    _gemini_model = genai.GenerativeModel('gemini-2.5-flash-lite')
    for attempt in range(3):
        try:
            response = _gemini_model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(max_output_tokens=1500)
            )
            return response.text
        except Exception as e:
            err = str(e)
            if '429' in err and attempt < 2:
                time.sleep(15)
            elif attempt == 2:
                raise RuntimeError(f'Gemini API error after 3 attempts: {err}')
            else:
                raise RuntimeError(f'Gemini API error: {err}')

# ── Full inference pipeline ───────────────────────────────────
def run_inference(video_path, filename, api_key=""):
    """Returns a dict with all results or raises an exception.
    Hard timeout is enforced by the Flask worker thread contract.
    """
    t0 = time.time()

    def elapsed():
        return time.time() - t0

    def check_timeout(stage):
        if elapsed() > UPLOAD_TIMEOUT_S:
            raise TimeoutError(f'Inference timed out at stage: {stage} ({elapsed():.0f}s elapsed, limit={UPLOAD_TIMEOUT_S}s)')

    frames    = extract_frames(video_path);    check_timeout('frame extraction')
    audio_p, has_audio = extract_audio(video_path)
    audio_np  = process_audio(audio_p, has_audio, video_path); check_timeout('audio processing')

    frames_t = (torch.from_numpy(np.array(frames))
                .permute(0,3,1,2).float().unsqueeze(0) / 255.0).to(device)
    audio_t  = (torch.from_numpy(audio_np)
                .unsqueeze(1).float().unsqueeze(0)).to(device)

    with torch.no_grad():
        logits = model(frames_t, audio_t)
        probs  = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
        pred   = int(probs.argmax())
    check_timeout('model inference')

    gradcams     = compute_gradcam(model, frames_t, audio_t, pred);  check_timeout('gradcam')
    temp_attn    = compute_temporal_attention(model, frames_t, audio_t); check_timeout('temporal attention')
    audio_sal    = compute_audio_saliency(audio_np)

    xai_b64 = generate_xai_image(frames, gradcams, temp_attn, audio_sal, probs, pred, filename)
    check_timeout('xai visualization')

    explanation = explain_with_gemini(probs, pred, temp_attn, audio_sal, gradcams, filename, api_key=api_key)
    check_timeout('gemini report')

    video_fake_prob = float(probs[1] + probs[3])
    audio_fake_prob = float(probs[2] + probs[3])
    av_sync_mismatch = (1 - float(probs[0])) * 100

    return {
        'prediction': CLASS_NAMES[pred],
        'pred_idx': pred,
        'confidence': float(probs[pred]) * 100,
        'is_fake': pred != 0,
        'has_audio': has_audio,
        'probabilities': [
            {'label': CLASS_NAMES[i], 'prob': float(probs[i]) * 100}
            for i in range(4)
        ],
        'xai_image_b64': xai_b64,
        'forensic_report': explanation,
        'metrics': {
            'video_fake_pct': round(video_fake_prob * 100, 1),
            'audio_fake_pct': round(audio_fake_prob * 100, 1),
            'av_sync_mismatch_pct': round(av_sync_mismatch, 1),
        },
        'processing_time_s': round(elapsed(), 1),
        'filename': filename
    }

print('✅ Inference functions ready')

✅ Inference functions ready


In [14]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Flask app + ngrok tunnel
# ─────────────────────────────────────────────────────────────
import threading, uuid, traceback
from flask import Flask, request, jsonify, render_template_string, send_from_directory
from flask_cors import CORS
from pyngrok import ngrok, conf
from werkzeug.utils import secure_filename

ALLOWED_EXTS = {'mp4', 'avi', 'mov', 'mkv', 'webm', 'flv'}
UPLOAD_DIR   = '/tmp/deepfake_uploads'
os.makedirs(UPLOAD_DIR, exist_ok=True)

app = Flask(__name__)
CORS(app)
app.config['MAX_CONTENT_LENGTH'] = MAX_VIDEO_MB * 1024 * 1024

# Thread-safe job store  {job_id: {status, result, error}}
jobs      = {}
jobs_lock = threading.Lock()

# ── HTML template ─────────────────────────────────────────────
HTML = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Multimodal Deepfake Detection</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@400;600;700;800&display=swap" rel="stylesheet">
<style>
  :root {
    --bg:      #07080f;
    --card:    #0e0f1a;
    --border:  #1a1b2e;
    --accent:  #5b5ef4;
    --accent2: #a259f7;
    --green:   #10d98a;
    --red:     #f04b5a;
    --text:    #dde1f0;
    --muted:   #7880a0;
    --radius:  14px;
    --mono:    'Space Mono', monospace;
    --sans:    'Syne', system-ui, sans-serif;
  }
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { background: var(--bg); color: var(--text); font-family: var(--sans);
         min-height: 100vh; display: flex; flex-direction: column; }
  header {
    background: linear-gradient(135deg, #08091a 0%, #0f1028 100%);
    border-bottom: 1px solid var(--border);
    padding: 18px 40px;
    display: flex; align-items: center; gap: 18px;
    position: sticky; top: 0; z-index: 100;
    flex-wrap: wrap;
  }
  .hdr-logo { font-size: 28px; flex-shrink: 0; }
  .hdr-title { flex: 1; min-width: 200px; }
  .hdr-title h1 {
    font-size: 1.1rem; font-weight: 800; letter-spacing: -0.02em;
    background: linear-gradient(90deg, var(--accent), var(--accent2));
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
  }
  .hdr-title p { font-size: 0.7rem; color: var(--muted); margin-top: 2px; }
  .hdr-authors { display: flex; gap: 7px; flex-wrap: wrap; }
  .author-pill {
    background: rgba(91,94,244,0.12); border: 1px solid rgba(91,94,244,0.28);
    color: #a5a9f8; border-radius: 999px; padding: 3px 10px;
    font-family: 'Space Mono', monospace; font-size: 0.68rem; white-space: nowrap;
  }
  .badge {
    background: rgba(162,89,247,0.14); border: 1px solid rgba(162,89,247,0.4);
    color: #c084fc; border-radius: 999px; padding: 4px 12px; font-size: 0.7rem;
    font-weight: 700; letter-spacing: 0.05em; flex-shrink: 0;
  }
  main { flex: 1; max-width: 1300px; margin: 0 auto; padding: 28px 24px; width: 100%; }
  .grid2 { display: grid; grid-template-columns: 1fr 1fr; gap: 22px; }
  @media (max-width: 860px) { .grid2 { grid-template-columns: 1fr; } }
  .card { background: var(--card); border: 1px solid var(--border); border-radius: var(--radius); padding: 22px; }
  .card-title { font-size: 0.78rem; font-weight: 700; margin-bottom: 16px; display: flex; align-items: center; gap: 8px; letter-spacing: 0.06em; text-transform: uppercase; color: var(--muted); }
  .upload-zone { border: 2px dashed var(--border); border-radius: var(--radius); padding: 34px 20px; text-align: center; cursor: pointer; transition: all .2s; position: relative; }
  .upload-zone:hover, .upload-zone.drag { border-color: var(--accent); background: rgba(91,94,244,0.05); }
  .upload-zone input { position: absolute; inset: 0; opacity: 0; cursor: pointer; width: 100%; height: 100%; }
  .upload-icon { font-size: 36px; margin-bottom: 10px; }
  .upload-zone p { color: var(--muted); font-size: 0.83rem; }
  .upload-zone strong { color: var(--text); }
  #file-name { margin-top: 10px; color: var(--accent); font-size: 0.8rem; font-weight: 600; font-family: 'Space Mono', monospace; }
  .btn { display: inline-flex; align-items: center; gap: 8px; padding: 11px 22px; border: none; border-radius: 8px; font-size: 0.9rem; font-weight: 700; font-family: 'Syne', system-ui, sans-serif; cursor: pointer; transition: all .15s; }
  .btn-primary { background: linear-gradient(135deg, var(--accent), var(--accent2)); color: white; width: 100%; justify-content: center; margin-top: 14px; }
  .btn-primary:hover { opacity: 0.88; transform: translateY(-1px); }
  .btn-primary:disabled { opacity: 0.35; cursor: not-allowed; transform: none; }
  .metrics-table { width: 100%; border-collapse: collapse; font-size: 0.82rem; }
  .metrics-table th { color: var(--muted); font-weight: 600; text-align: left; padding: 7px 10px; border-bottom: 1px solid var(--border); font-family: 'Space Mono', monospace; font-size: 0.68rem; letter-spacing: 0.05em; }
  .metrics-table td { padding: 7px 10px; border-bottom: 1px solid rgba(26,27,46,0.5); }
  .metrics-table tr:last-child td { border-bottom: none; }
  .metrics-table .avg-row td { font-weight: 700; color: var(--accent); }
  .pill { display: inline-block; padding: 2px 8px; border-radius: 999px; font-size: 0.7rem; font-weight: 700; }
  .pill-green { background: rgba(16,217,138,0.12); color: var(--green); }
  .prob-bar { display: flex; align-items: center; gap: 10px; margin: 8px 0; font-size: 0.82rem; }
  .prob-bar .label { min-width: 170px; font-size: 0.76rem; color: var(--muted); }
  .prob-bar .track { flex: 1; height: 7px; background: var(--border); border-radius: 4px; overflow: hidden; }
  .prob-bar .fill { height: 100%; border-radius: 4px; background: linear-gradient(90deg, var(--accent), var(--accent2)); transition: width .6s; }
  .prob-bar .fill.active { background: linear-gradient(90deg, var(--green), #4ade80); }
  .prob-bar .fill.danger { background: linear-gradient(90deg, var(--red), #f97316); }
  .prob-pct { min-width: 40px; text-align: right; color: var(--muted); font-family: 'Space Mono', monospace; font-size: 0.76rem; }
  .verdict { border-radius: var(--radius); padding: 18px 22px; margin-bottom: 20px; display: flex; align-items: center; gap: 16px; }
  .verdict.safe { background: rgba(16,217,138,0.08); border: 1px solid rgba(16,217,138,0.28); }
  .verdict.danger { background: rgba(240,75,90,0.08); border: 1px solid rgba(240,75,90,0.28); }
  .verdict-emoji { font-size: 32px; flex-shrink: 0; }
  .verdict-title { font-size: 1.08rem; font-weight: 800; }
  .verdict-sub { font-size: 0.78rem; color: var(--muted); margin-top: 3px; font-family: 'Space Mono', monospace; }
  .chips { display: flex; gap: 10px; flex-wrap: wrap; margin-bottom: 18px; }
  .metric-chip { display: inline-flex; align-items: center; gap: 6px; background: var(--card); border: 1px solid var(--border); border-radius: 9px; padding: 9px 14px; font-size: 0.78rem; }
  .metric-chip .val { font-size: 1.05rem; font-weight: 700; font-family: 'Space Mono', monospace; }
  .tab-bar { display: flex; gap: 4px; margin-bottom: 20px; border-bottom: 1px solid var(--border); }
  .tab-btn { background: transparent; border: none; color: var(--muted); font-family: 'Syne', system-ui, sans-serif; font-size: 0.88rem; font-weight: 700; padding: 10px 20px; cursor: pointer; border-bottom: 3px solid transparent; margin-bottom: -1px; transition: all .15s; border-radius: 6px 6px 0 0; display: flex; align-items: center; gap: 7px; }
  .tab-btn:hover { color: var(--text); background: rgba(91,94,244,0.06); }
  .tab-btn.active { color: var(--accent); border-bottom-color: var(--accent); background: rgba(91,94,244,0.07); }
  .tab-pane { display: none; }
  .tab-pane.active { display: block; }
  #xai-img { width: 100%; border-radius: var(--radius); border: 1px solid var(--border); margin-top: 8px; display: block; cursor: zoom-in; max-height: 700px; object-fit: contain; }
  #xai-img.zoomed { max-height: none; cursor: zoom-out; }
  .xai-hint { font-size: 0.72rem; color: var(--muted); margin-top: 6px; text-align: center; }
  .report-text { background: #06070f; border: 1px solid var(--border); border-radius: 10px; padding: 22px; font-size: 0.9rem; line-height: 1.85; color: #c8d0e8; white-space: pre-wrap; font-family: 'Syne', system-ui, sans-serif; min-height: 300px; }
  .arch-row { display: flex; align-items: center; gap: 7px; margin: 6px 0; flex-wrap: wrap; }
  .arch-box { background: rgba(91,94,244,0.09); border: 1px solid rgba(91,94,244,0.26); border-radius: 7px; padding: 7px 12px; font-size: 0.72rem; color: #a5a9f8; font-family: 'Space Mono', monospace; line-height: 1.5; }
  .arch-arr { color: var(--muted); font-size: 0.85rem; }
  #status { display: none; text-align: center; padding: 48px 32px; }
  .spinner { width: 48px; height: 48px; border: 3px solid var(--border); border-top-color: var(--accent); border-radius: 50%; animation: spin 0.75s linear infinite; margin: 0 auto 16px; }
  @keyframes spin { to { transform: rotate(360deg); } }
  #status p { color: var(--muted); font-size: 0.86rem; }
  #status .stage { color: var(--accent); font-weight: 700; margin-top: 6px; font-family: 'Space Mono', monospace; font-size: 0.8rem; }
  .api-key-row { margin-top: 14px; }
  .api-key-label { display: block; font-size: 0.72rem; color: var(--muted); font-weight: 700;
                   letter-spacing: 0.06em; text-transform: uppercase; margin-bottom: 6px; }
  .api-key-input { width: 100%; background: #06070f; border: 1px solid var(--border);
                   border-radius: 8px; padding: 9px 13px; color: var(--text);
                   font-family: 'Space Mono', monospace; font-size: 0.8rem;
                   outline: none; transition: border-color .15s; }
  .api-key-input:focus { border-color: var(--accent); }
  .api-key-input::placeholder { color: var(--muted); }
  .api-key-link { display: inline-block; margin-top: 5px; font-size: 0.72rem;
                  color: var(--accent); text-decoration: none; }
  .api-key-link:hover { text-decoration: underline; }
  .error-box { background: rgba(240,75,90,0.08); border: 1px solid rgba(240,75,90,0.3); border-radius: var(--radius); padding: 14px; color: #fca5a5; font-size: 0.84rem; display: none; margin-top: 14px; }
  #results { display: none; margin-top: 22px; }
  footer { background: #06070f; border-top: 1px solid var(--border); padding: 22px 40px; margin-top: 40px; }
  .footer-inner { max-width: 1300px; margin: 0 auto; display: flex; flex-wrap: wrap; gap: 22px; align-items: flex-start; justify-content: space-between; }
  .footer-team h3 { font-size: 0.68rem; color: var(--muted); font-weight: 700; letter-spacing: 0.1em; text-transform: uppercase; margin-bottom: 10px; }
  .footer-members { display: flex; gap: 10px; flex-wrap: wrap; }
  .member-card { background: var(--card); border: 1px solid var(--border); border-radius: 9px; padding: 8px 14px; }
  .member-name { font-size: 0.82rem; font-weight: 700; color: var(--text); }
  .member-role { font-size: 0.68rem; color: var(--muted); margin-top: 2px; }
  .footer-stats { display: flex; gap: 12px; flex-wrap: wrap; align-items: center; }
  .stat-badge { background: var(--card); border: 1px solid var(--border); border-radius: 9px; padding: 8px 14px; text-align: center; font-family: 'Space Mono', monospace; }
  .stat-val { font-size: 1rem; font-weight: 700; color: var(--accent); }
  .stat-label { font-size: 0.65rem; color: var(--muted); margin-top: 2px; }
  .footer-copy { font-size: 0.68rem; color: var(--muted); margin-top: 12px; width: 100%; }
</style>
</head>
<body>

<header>
  <div class="hdr-logo">🧠</div>
  <div class="hdr-title">
    <h1>Multimodal Deepfake Detection</h1>
    <p>Cross-Modal Attention + Explainable AI &nbsp;·&nbsp; SRMIST Ramapuram &nbsp;·&nbsp; ICRETEM 2026</p>
  </div>
  <div class="hdr-authors">
    <span class="author-pill">Thilak Raaj N V</span>
    <span class="author-pill">Samuel Raj Irwin V</span>
    <span class="author-pill">Balasudhan C M</span>
    <span class="author-pill" style="background:rgba(162,89,247,0.1);border-color:rgba(162,89,247,0.3);color:#c084fc;">Dr. Sujatha K</span>
  </div>
  <div class="badge">SRMIST Ramapuram</div>
</header>

<main>

<!-- Upload + Metrics row -->
<div class="grid2" style="margin-bottom:22px">

  <div class="card">
    <div class="card-title"><span>📤</span> Analyse a Video</div>
    <div class="upload-zone" id="drop-zone">
      <input type="file" id="file-input" accept="video/*">
      <div class="upload-icon">🎬</div>
      <p><strong>Click or drag &amp; drop</strong> a video file</p>
      <p style="margin-top:4px">MP4, AVI, MOV, MKV, WEBM &nbsp;·&nbsp; Max {{ max_mb }} MB</p>
      <div id="file-name"></div>
    </div>
    <div class="api-key-row">
      <label class="api-key-label">🔑 Gemini API Key</label>
      <input type="password" id="gemini-key" class="api-key-input"
             placeholder="Paste your key from aistudio.google.com" autocomplete="off">
      <a href="https://aistudio.google.com/app/apikey" target="_blank" class="api-key-link">Get a free key ↗</a>
    </div>
    <button class="btn btn-primary" id="analyse-btn" disabled>
      <span>🔍</span> Analyse for Deepfake
    </button>
    <div class="error-box" id="error-box"></div>
  </div>

  <div class="card">
    <div class="card-title"><span>📊</span> Published Results</div>
    <table class="metrics-table">
      <thead><tr>
        <th>Dataset</th><th>Accuracy</th><th>Macro-AUC</th><th>EER</th>
      </tr></thead>
      <tbody id="metrics-body"></tbody>
    </table>
    <div style="margin-top:16px">
      <div style="font-size:0.68rem;color:var(--muted);margin-bottom:10px;font-family:'Space Mono',monospace;letter-spacing:0.05em">PER-CLASS F1 · FAKEAVCELEB</div>
      <div id="f1-bars"></div>
    </div>
  </div>
</div>

<!-- Architecture (always visible, before any upload) -->
<div class="card" style="margin-bottom:22px">
  <div class="card-title"><span>⚙️</span> Model Architecture</div>
  <div class="arch-row">
    <div class="arch-box">ResNet18<br><span style="color:var(--muted);font-size:0.65rem">Visual Encoder · 512d</span></div>
    <span class="arch-arr">→</span>
    <div class="arch-box">Temporal Transformer<br><span style="color:var(--muted);font-size:0.65rem">2L · 8H · d_model=512</span></div>
    <span class="arch-arr">→</span>
    <div class="arch-box" style="border-color:rgba(162,89,247,0.35);color:#d8b4fe">Cross-Modal Attention<br><span style="color:var(--muted);font-size:0.65rem">Visual(Q) × Audio(K,V)</span></div>
    <span class="arch-arr">→</span>
    <div class="arch-box" style="border-color:rgba(16,217,138,0.3);color:#6ee7b7">AV Sync Feature<br><span style="color:var(--muted);font-size:0.65rem">Cosine Sim Scalar</span></div>
    <span class="arch-arr">→</span>
    <div class="arch-box" style="border-color:rgba(251,191,36,0.3);color:#fde68a">4-Class Classifier<br><span style="color:var(--muted);font-size:0.65rem">FC(512→512→4)</span></div>
  </div>
  <div class="arch-row" style="margin-top:10px">
    <div class="arch-box" style="border-color:rgba(16,217,138,0.3);color:#6ee7b7">Audio CNN<br><span style="color:var(--muted);font-size:0.65rem">3-Layer · Mel Spectrogram</span></div>
    <span class="arch-arr">→</span>
    <div class="arch-box" style="border-color:rgba(16,217,138,0.3);color:#6ee7b7">Audio Embedding<br><span style="color:var(--muted);font-size:0.65rem">256d → projected 512d</span></div>
    <span class="arch-arr">↗</span>
    <span style="font-size:0.72rem;color:var(--muted);font-style:italic">feeds into Cross-Modal Attention above</span>
  </div>
  <div style="margin-top:12px;font-size:0.75rem;color:var(--muted);line-height:1.6">
    Trained jointly on <strong style="color:var(--text)">FakeAVCeleb + LAV-DF + FaceForensics++ C23</strong>
    &nbsp;·&nbsp; Focal Loss γ=2.0 · Label Smoothing 0.1 · AMP fp16 · Frame Dropout p=0.2 · Audio Dropout p=0.3 · Adam lr=1e-4
  </div>
</div>

<!-- Status spinner -->
<div id="status">
  <div class="spinner"></div>
  <p>Running deepfake analysis…</p>
  <p class="stage" id="stage-text">Initialising pipeline</p>
  <p style="color:var(--muted);font-size:0.74rem;margin-top:8px">This may take 1–3 minutes depending on video length.</p>
</div>

<!-- Results (tabbed) -->
<div id="results">

  <div id="verdict-banner" class="verdict"></div>
  <div class="chips" id="metric-chips"></div>

  <div class="tab-bar">
    <button class="tab-btn active" onclick="switchTab('detection',this)">
      <span>📈</span> Detection Results
    </button>
    <button class="tab-btn" onclick="switchTab('xai',this)">
      <span>🔬</span> XAI Visualisation
    </button>
    <button class="tab-btn" onclick="switchTab('report',this)">
      <span>🤖</span> Forensic Report
    </button>
  </div>

  <div class="tab-pane active" id="tab-detection">
    <div class="card">
      <div class="card-title"><span>📈</span> Class Probabilities</div>
      <div id="prob-bars"></div>
    </div>
  </div>

  <div class="tab-pane" id="tab-xai">
    <div class="card">
      <div class="card-title"><span>🔬</span> XAI Visualisation</div>
      <p style="font-size:0.76rem;color:var(--muted);margin-bottom:8px">
        Row 1: raw frames &nbsp;·&nbsp; Row 2: Grad-CAM overlays &nbsp;·&nbsp; Row 3: temporal attention + audio saliency
      </p>
      <img id="xai-img" alt="XAI visualization" onclick="this.classList.toggle('zoomed')">
      <p class="xai-hint">🔍 Click to zoom in/out</p>
    </div>
  </div>

  <div class="tab-pane" id="tab-report">
    <div class="card">
      <div class="card-title"><span>🤖</span> Gemini Forensic Report</div>
      <div class="report-text" id="report-text"></div>
    </div>
  </div>

</div>

</main>

<footer>
  <div class="footer-inner">
    <div class="footer-team">
      <h3>Research Team — SRMIST Ramapuram</h3>
      <div class="footer-members">
        <div class="member-card">
          <div class="member-name">Thilak Raaj N V</div>
          <div class="member-role">RA2211003020643</div>
        </div>
        <div class="member-card">
          <div class="member-name">Samuel Raj Irwin V</div>
          <div class="member-role">RA2211003020683</div>
        </div>
        <div class="member-card">
          <div class="member-name">Balasudhan C M</div>
          <div class="member-role">RA2211003020691</div>
        </div>
        <div class="member-card" style="border-color:rgba(162,89,247,0.3)">
          <div class="member-name" style="color:#c084fc">Dr. Sujatha K</div>
          <div class="member-role">Project Advisor</div>
        </div>
      </div>
    </div>
    <div class="footer-stats">
      <div class="stat-badge">
        <div class="stat-val">86.6%</div>
        <div class="stat-label">Avg Accuracy</div>
      </div>
      <div class="stat-badge">
        <div class="stat-val">0.980</div>
        <div class="stat-label">Macro-AUC</div>
      </div>
      <div class="stat-badge">
        <div class="stat-val">90.4%</div>
        <div class="stat-label">FakeAVCeleb</div>
      </div>
      <div class="stat-badge">
        <div class="stat-val" style="color:var(--green)">ICRETEM</div>
        <div class="stat-label">2026 · Coimbatore</div>
      </div>
    </div>
    <div class="footer-copy">
      © 2026 SRMIST Ramapuram &nbsp;·&nbsp; 6th International Conference on Recent Trends in Engineering, Technology and Management (ICRETEM 2026), Suguna College of Engineering, Coimbatore, India &nbsp;·&nbsp; Academic &amp; Research Use Only
    </div>
  </div>
</footer>

<script>
const METRICS = {{ metrics_json | safe }};

const tbody = document.getElementById('metrics-body');
METRICS.datasets.forEach(row => {
  const tr = document.createElement('tr');
  if (row.name === 'Average') tr.classList.add('avg-row');
  tr.innerHTML = `
    <td>${row.name}</td>
    <td><span class="pill pill-green">${row.accuracy}%</span></td>
    <td>${row.macro_auc !== null ? row.macro_auc.toFixed(3) : '—'}</td>
    <td>${row.eer !== null ? row.eer + '%' : '—'}</td>
  `;
  tbody.appendChild(tr);
});

const f1div = document.getElementById('f1-bars');
METRICS.f1_scores.forEach(r => {
  f1div.innerHTML += `
    <div class="prob-bar">
      <span class="label" style="font-size:0.73rem;min-width:155px">${r.class}</span>
      <div class="track"><div class="fill" style="width:${r.f1}%"></div></div>
      <span class="prob-pct">${r.f1}%</span>
    </div>`;
});

function switchTab(id, btn) {
  document.querySelectorAll('.tab-pane').forEach(p => p.classList.remove('active'));
  document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
  document.getElementById('tab-' + id).classList.add('active');
  btn.classList.add('active');
}

const dropZone  = document.getElementById('drop-zone');
const fileInput = document.getElementById('file-input');
const fileName  = document.getElementById('file-name');
const analyseBtn = document.getElementById('analyse-btn');
const errorBox  = document.getElementById('error-box');
let selectedFile = null;

function updateAnalyseBtn() {
  const hasFile = !!selectedFile;
  const hasKey  = document.getElementById('gemini-key').value.trim().length > 10;
  analyseBtn.disabled = !(hasFile && hasKey);
}
document.getElementById('gemini-key').addEventListener('input', updateAnalyseBtn);

['dragover','dragenter'].forEach(e => dropZone.addEventListener(e, ev => {
  ev.preventDefault(); dropZone.classList.add('drag');
}));
['dragleave','drop'].forEach(e => dropZone.addEventListener(e, ev => {
  ev.preventDefault(); dropZone.classList.remove('drag');
}));
dropZone.addEventListener('drop', ev => {
  const f = ev.dataTransfer.files[0];
  if (f) setFile(f);
});
fileInput.addEventListener('change', () => {
  if (fileInput.files[0]) setFile(fileInput.files[0]);
});

function setFile(f) {
  const maxBytes = {{ max_mb }} * 1024 * 1024;
  errorBox.style.display = 'none';
  if (f.size > maxBytes) {
    showError(`File too large: ${(f.size/1024/1024).toFixed(1)} MB (max {{ max_mb }} MB)`);
    return;
  }
  selectedFile = f;
  fileName.textContent = `📎 ${f.name}  (${(f.size/1024/1024).toFixed(1)} MB)`;
  updateAnalyseBtn();
}

function showError(msg) {
  errorBox.textContent = '⚠️  ' + msg;
  errorBox.style.display = 'block';
}

analyseBtn.addEventListener('click', async () => {
  if (!selectedFile) return;
  errorBox.style.display = 'none';
  document.getElementById('results').style.display = 'none';
  document.getElementById('status').style.display  = 'block';
  analyseBtn.disabled = true;

  const fd = new FormData();
  fd.append('video', selectedFile);
  fd.append('gemini_key', document.getElementById('gemini-key').value.trim());

  let jobId;
  try {
    const resp = await fetch('/analyse', { method: 'POST', body: fd });
    const data = await resp.json();
    if (!resp.ok || data.error) throw new Error(data.error || `HTTP ${resp.status}`);
    jobId = data.job_id;
  } catch (e) {
    document.getElementById('status').style.display = 'none';
    analyseBtn.disabled = false;
    showError('Upload failed: ' + e.message);
    return;
  }

  const stages = [
    'Extracting frames…', 'Processing audio…', 'Running model inference…',
    'Computing Grad-CAM…', 'Generating temporal attention…',
    'Building XAI visualisation…', 'Generating Gemini forensic report…'
  ];
  let si = 0;
  const stageEl = document.getElementById('stage-text');
  const stageTimer = setInterval(() => {
    stageEl.textContent = stages[si % stages.length]; si++;
  }, 4000);

  const POLL_INTERVAL = 2000;
  const MAX_WAIT = 360000;
  const deadline = Date.now() + MAX_WAIT;

  while (Date.now() < deadline) {
    await new Promise(r => setTimeout(r, POLL_INTERVAL));
    let poll;
    try {
      const r = await fetch(`/status/${jobId}`);
      poll = await r.json();
    } catch {
      continue;
    }

    if (poll.status === 'done') {
      clearInterval(stageTimer);
      document.getElementById('status').style.display = 'none';
      analyseBtn.disabled = false;
      renderResults(poll.result);
      return;
    }
    if (poll.status === 'error') {
      clearInterval(stageTimer);
      document.getElementById('status').style.display = 'none';
      analyseBtn.disabled = false;
      showError(poll.error);
      return;
    }
  }

  clearInterval(stageTimer);
  document.getElementById('status').style.display = 'none';
  analyseBtn.disabled = false;
  showError('Timed out waiting for result. The video may be too long or the server is under load.');
});

function renderResults(r) {
  const banner = document.getElementById('verdict-banner');
  banner.className = 'verdict ' + (r.is_fake ? 'danger' : 'safe');
  banner.innerHTML = `
    <div class="verdict-emoji">${r.is_fake ? '🚨' : '✅'}</div>
    <div>
      <div class="verdict-title">${r.prediction}</div>
      <div class="verdict-sub">
        Confidence: <strong>${r.confidence.toFixed(1)}%</strong> &nbsp;|&nbsp;
        File: ${r.filename} &nbsp;|&nbsp;
        Processed in ${r.processing_time_s}s
        ${!r.has_audio ? ' &nbsp;|&nbsp; ⚠️ No audio track — video-only prediction' : ''}
      </div>
    </div>`;

  const chips = document.getElementById('metric-chips');
  chips.innerHTML = [
    ['🎥 Video Fake Prob', r.metrics.video_fake_pct + '%', r.metrics.video_fake_pct > 50 ? '#f04b5a' : '#10d98a'],
    ['🎙️ Audio Fake Prob', r.metrics.audio_fake_pct + '%', r.metrics.audio_fake_pct > 50 ? '#f04b5a' : '#10d98a'],
    ['🔄 AV Sync Mismatch', r.metrics.av_sync_mismatch_pct + '%', r.metrics.av_sync_mismatch_pct > 50 ? '#f04b5a' : '#10d98a'],
    ['⏱️ Processing Time', r.processing_time_s + 's', '#818cf8'],
  ].map(([label, val, color]) =>
    `<div class="metric-chip">${label} <span class="val" style="color:${color}">${val}</span></div>`
  ).join('');

  const pbDiv = document.getElementById('prob-bars');
  pbDiv.innerHTML = r.probabilities.map((p, i) => {
    const isActive = i === r.pred_idx;
    const cls = isActive ? (r.is_fake ? 'danger' : 'active') : '';
    return `<div class="prob-bar">
      <span class="label">${isActive ? '▶ ' : ''}${p.label}</span>
      <div class="track"><div class="fill ${cls}" style="width:${p.prob}%"></div></div>
      <span class="prob-pct">${p.prob.toFixed(1)}%</span>
    </div>`;
  }).join('');

  const xaiImg = document.getElementById('xai-img');
  xaiImg.src = 'data:image/png;base64,' + r.xai_image_b64;
  xaiImg.classList.remove('zoomed');

  document.getElementById('report-text').textContent = r.forensic_report;

  document.getElementById('results').style.display = 'block';
  switchTab('detection', document.querySelectorAll('.tab-btn')[0]);
  document.getElementById('results').scrollIntoView({ behavior: 'smooth' });
}
</script>
</body>
</html>

"""

# ── Flask routes ──────────────────────────────────────────────
@app.route('/')
def index():
    import json as _json
    return render_template_string(
        HTML,
        max_mb=MAX_VIDEO_MB,
        metrics_json=_json.dumps(PAPER_METRICS)
    )

@app.route('/analyse', methods=['POST'])
def analyse():
    if 'video' not in request.files:
        return jsonify({'error': 'No video file provided'}), 400

    f = request.files['video']
    if f.filename == '':
        return jsonify({'error': 'Empty filename'}), 400

    ext = f.filename.rsplit('.', 1)[-1].lower() if '.' in f.filename else ''
    if ext not in ALLOWED_EXTS:
        return jsonify({'error': f'Unsupported format: .{ext}. Allowed: {ALLOWED_EXTS}'}), 400

    job_id   = str(uuid.uuid4())
    safe_fn  = secure_filename(f.filename)
    save_path = os.path.join(UPLOAD_DIR, f'{job_id}_{safe_fn}')
    api_key  = request.form.get('gemini_key', '').strip()
    if not api_key:
        return jsonify({'error': 'Gemini API key is required. Please enter it in the web UI.'}), 400
    f.save(save_path)

    with jobs_lock:
        jobs[job_id] = {'status': 'running', 'result': None, 'error': None}

    # Run inference in a daemon thread so Flask is never blocked
    def worker():
        try:
            result = run_inference(save_path, safe_fn, api_key=api_key)
            with jobs_lock:
                jobs[job_id]['status'] = 'done'
                jobs[job_id]['result'] = result
        except TimeoutError as e:
            with jobs_lock:
                jobs[job_id]['status'] = 'error'
                jobs[job_id]['error']  = f'Timeout: {e}'
        except Exception as e:
            tb = traceback.format_exc()
            print(f'[ERROR] job {job_id}: {e}\n{tb}')
            with jobs_lock:
                jobs[job_id]['status'] = 'error'
                jobs[job_id]['error']  = str(e)
        finally:
            # Clean up uploaded file
            try:
                os.remove(save_path)
            except Exception:
                pass

    t = threading.Thread(target=worker, daemon=True)
    t.start()

    return jsonify({'job_id': job_id}), 202

@app.route('/status/<job_id>')
def status(job_id):
    with jobs_lock:
        job = jobs.get(job_id)
    if job is None:
        return jsonify({'error': 'Unknown job ID'}), 404
    # Return status; strip large b64 from pending jobs to keep polling cheap
    return jsonify({
        'status': job['status'],
        'result': job['result'],   # None while running
        'error':  job['error']
    })

@app.errorhandler(413)
def too_large(e):
    return jsonify({'error': f'File exceeds {MAX_VIDEO_MB} MB limit'}), 413

# ── Launch ngrok + Flask ──────────────────────────────────────
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Close any existing ngrok tunnels
ngrok.kill()

PORT = 5000
public_url = ngrok.connect(PORT)
print('=' * 60)
print(f'🌐  PUBLIC URL : {public_url}')
print(f'🔒  Local port : http://localhost:{PORT}')
print('=' * 60)
print('Share the public URL above to access the web UI from anywhere.')
print('Ctrl+C to stop the server.')

# Run Flask in the main thread (Colab-friendly)
app.run(port=PORT, debug=False, use_reloader=False)

🌐  PUBLIC URL : NgrokTunnel: "https://grumble-backed-sublease.ngrok-free.dev" -> "http://localhost:5000"
🔒  Local port : http://localhost:5000
Share the public URL above to access the web UI from anywhere.
Ctrl+C to stop the server.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [06/May/2026 07:23:29] "GET / HTTP/1.1" 200 -
